In [1]:
import subprocess 
import subprocess
import sys

import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()
import subprocess
import sys

subprocess.run('tar --no-same-permissions -cf - -C /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script . | tar -xf - -C /tmp', shell=True, check=False)
subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas', shell=True)
subprocess.run('chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell', shell=True)
sys.path.insert(0, '/tmp')

In [2]:
import subprocess
import sys

# Install from local datasets (no internet needed)
subprocess.run(
    "pip install -q --no-index --find-links /kaggle/input/nemotron-packages/packages "
    "unsloth trl peft transformers datasets accelerate bitsandbytes",
    shell=True
)
subprocess.run(
    "pip install -q /kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True
)
subprocess.run(
    "pip install -q /kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True
)

ERROR: Could not find a version that satisfies the requirement unsloth (from versions: none)
ERROR: No matching distribution found for unsloth
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'



CompletedProcess(args='pip install -q /kaggle/input/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl', returncode=1)

In [4]:
# import site
# import os
# import kagglehub
# import mamba_ssm
# import torch
# from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
# from transformers import AutoModelForCausalLM, AutoTokenizer


# import os
# import site
# import subprocess

# import kagglehub
# import pandas as pd
# import torch
# from peft import LoraConfig, TaskType, get_peft_model
# from torch.utils.data import Dataset
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     Trainer,
#     TrainingArguments,
# )


# os.environ["TRANSFORMERS_NO_TF"] = "1"
# os.environ["TRANSFORMERS_NO_FLAX"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

# utility_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"
# if os.path.exists(utility_path):
#     subprocess.run(
#         f"tar --no-same-permissions -cf - -C {utility_path} . | tar -xf - -C /tmp",
#         shell=True,
#         check=False,
#     )
#     subprocess.run(
#         "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
#         shell=True,
#         check=False,
#     )
#     subprocess.run(
#         "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
#         shell=True,
#         check=False,
#     )

# cutlass_pkg_path = (
#     f"{utility_path}/nvidia_cutlass_dsl/python_packages/"
# )
# site.addsitedir(cutlass_pkg_path)

# try:
#     import mamba_ssm  # noqa: F401
# except ImportError:
#     pass


# MODEL_PATH = kagglehub.model_download(
#     "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
# )
# TRAIN_PATH = "/kaggle/working/train.csv"
# OUTPUT_DIR = "/kaggle/working/nemotron_lora_adapter"

# LORA_RANK = 32
# MAX_LENGTH = 1024
# EPOCHS = 20
# LEARNING_RATE = 2e-4


# class ReasoningDataset(Dataset):
#     def __init__(self, dataframe, tokenizer, max_length):
#         self.rows = dataframe.reset_index(drop=True) # this creates a new DataFrame with a new index starting from 0, and drops the old index
#         self.tokenizer = tokenizer # get the tokenizer to tokenize the prompts and answers
#         self.max_length = max_length # the maximum length of the input sequence, including the prompt and the answer

#     def __len__(self):
#         return len(self.rows) # get the number of rows in the dataframe, which is the number of samples in the dataset

#     def __getitem__(self, idx):
#         row = self.rows.iloc[idx] # get the row at the given index
#         prompt = str(row["prompt"]).strip() # get the prompt from the row and strip to not remove the space inside the prompt
#         answer = str(row["answer"]).strip() # get the answer from the row and strip to not remove the space inside the answer

#         if not (answer.startswith("\\boxed{") and answer.endswith("}")):
#             answer = f"\\boxed{{{answer}}}"

#         user_text = (
#             "Solve the problem. Return only the final answer inside \\boxed{}.\n\n"
#             f"{prompt}\n\nAnswer:"
#         )
#         full_text = f"{user_text} {answer}{self.tokenizer.eos_token}" # add the eos token at the end of the answer to indicate the end of the sequence

#         prompt_token_count = len( # get the number of tokens in the prompt to know which tokens to ignore in the loss calculation
#             self.tokenizer(
#                 user_text,
#                 truncation=True, # truncate the prompt if it's too long
#                 max_length=self.max_length, 
#                 add_special_tokens=True, # add special tokens to the prompt if the model uses them
#             )["input_ids"] # get the input ids of the prompt to count the number of tokens in the prompt
#             # example: [1, 2, 3, 4, 5]
#         )
#         full = self.tokenizer( # tokenize the full text (prompt + answer) to get the input ids and attention mask for the model
#             full_text,
#             truncation=True,
#             max_length=self.max_length,
#             padding="max_length",
#             return_tensors="pt",
#         )

#         input_ids = full["input_ids"][0] # get the input ids of the full text, and remove the batch dimension
#         attention_mask = full["attention_mask"][0] # get the attention mask of the full text, and remove the batch dimension
#         labels = input_ids.clone() # create a copy of the input ids to use as labels for the loss calculation

#         labels[:prompt_token_count] = -100 # set the labels of the prompt tokens to -100 to ignore them in the loss calculation
#         labels[attention_mask == 0] = -100 # set the labels of the padding tokens to -100 to ignore them in the loss calculation

#         return {
#             "input_ids": input_ids,
#             "attention_mask": attention_mask,
#             "labels": labels,
#         }


# def main():
#     if LORA_RANK > 32:
#         raise ValueError("LORA_RANK must be <= 32 for this challenge.")

#     model = AutoModelForCausalLM.from_pretrained(
#         MODEL_PATH,
#         device_map="auto",
#         trust_remote_code=True,
#         dtype=torch.bfloat16,
#     )
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     lora_config = LoraConfig(
#         r=LORA_RANK,
#         lora_alpha=16,
#         target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
#         lora_dropout=0.05,
#         bias="none",
#         task_type=TaskType.CAUSAL_LM,
#     )
#     model = get_peft_model(model, lora_config)
#     model.config.use_cache = False
#     model.print_trainable_parameters()

#     df = pd.read_csv(TRAIN_PATH)
#     df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
#     split_idx = int(len(df) * 0.80)
#     train_df = df.iloc[:split_idx].copy()
#     eval_df = df.iloc[split_idx:].copy()

#     train_dataset = ReasoningDataset(train_df, tokenizer, MAX_LENGTH)
#     eval_dataset = ReasoningDataset(eval_df, tokenizer, MAX_LENGTH)

#     try:
#         import bitsandbytes  # noqa: F401

#         optimizer_name = "paged_adamw_8bit"
#     except ImportError:
#         optimizer_name = "adamw_torch"

#     args_kwargs = {
#         "output_dir": "/kaggle/working/nemotron_training_logs",
#         "num_train_epochs": EPOCHS,
#         "per_device_train_batch_size": 1,
#         "per_device_eval_batch_size": 1,
#         "gradient_accumulation_steps": 8,
#         "learning_rate": LEARNING_RATE,
#         "bf16": True,
#         "optim": optimizer_name,
#         "logging_strategy": "steps",
#         "logging_steps": 10,
#         "save_strategy": "epoch",
#         "report_to": "none",
#         "remove_unused_columns": False,
#         "gradient_checkpointing": True,
#     }
#     if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
#         args_kwargs["eval_strategy"] = "epoch"
#     else:
#         args_kwargs["evaluation_strategy"] = "epoch"
#     training_args = TrainingArguments(**args_kwargs)

#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=eval_dataset,
#     )

#     trainer.train()
#     metrics = trainer.evaluate()
#     print(metrics)

#     model.save_pretrained(OUTPUT_DIR)
#     tokenizer.save_pretrained(OUTPUT_DIR)
#     print(f"Saved LoRA adapter to: {OUTPUT_DIR}")


# if __name__ == "__main__":
#     main()





















# # Configuration
# MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
# OUTPUT_DIR = "/kaggle/working"
# LORA_RANK = 32  # Can be set to a maximum of 32

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     device_map="auto",
#     trust_remote_code=True,
#     dtype=torch.bfloat16
# )
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
# print("Model loaded successfully.")

# # Initialize LoRA Adapter
# print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
# lora_config = LoraConfig(
#     r=LORA_RANK,
#     lora_alpha=16,
#     target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
#     lora_dropout=0.05,
#     bias="none",
#     task_type=TaskType.CAUSAL_LM,
# )


# # # Apply LoRA to the model
# # model = get_peft_model(model, lora_config)
# # model.print_trainable_parameters()


# # # YOUR CODE HERE
# # # --------------
# # # model.train() 
# # # --------------

# inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
# output = model.generate(**inputs, max_new_tokens=50)
# print(tokenizer.decode(output[0], skip_special_tokens=True))# # Save Adapter
# # print(f"Saving adapter to {OUTPUT_DIR}...")
# # model.save_pretrained(OUTPUT_DIR)

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


Model loaded successfully.
Initializing LoRA adapter with rank=32...
Hello World program in Rust

fn main() {
    println!("Hello, world!");
}
```


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/logs

In [ ]:
import os, site, subprocess, importlib.util, kagglehub
import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

# ── env setup ────────────────────────────────────────────────────────────────
os.environ.update({
    "TRANSFORMERS_NO_TF": "1", # Disable TensorFlow to avoid unnecessary imports and potential conflicts
    "TRANSFORMERS_NO_FLAX": "1", # Disable Flax to avoid unnecessary imports and potential conflicts
    "CUDA_VISIBLE_DEVICES": "0", # Use only the first GPU (if multiple are available) to avoid out-of-memory errors
    "TRITON_PTXAS_PATH": "/tmp/triton/backends/nvidia/bin/ptxas", # Set the path to the ptxas binary for NVIDIA Triton
    # ptxas is a tool used by NVIDIA Triton to compile CUDA code for GPU execution.
}) 
_util = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script" # Path to the NVIDIA utility script that contains necessary files for 
# NVIDIA Triton, which is a high-performance inference server that can be used to deploy machine learning models on NVIDIA GPUs
if os.path.exists(_util):
    for cmd in [
        f"tar --no-same-permissions -cf - -C {_util} . | tar -xf - -C /tmp",
        "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
        "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
    ]:
        subprocess.run(cmd, shell=True, check=False)
    site.addsitedir(f"{_util}/nvidia_cutlass_dsl/python_packages/") # Add the NVIDIA Cutlass DSL Python packages to the site directories for import

try:
    import mamba_ssm  # mamba is a library for efficient sequence modeling, and mamba_ssm is a specific implementation of it that can be used for training
    
except ImportError:
    pass

# ── config ───────────────────────────────────────────────────────────────────
MODEL_PATH  = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
TRAIN_PATH  = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
OUTPUT_DIR  = "/kaggle/working/nemotron_lora_adapter"
METRIC_PATH = "/kaggle/usr/lib/notebooks/metric/nvidia-nemotron-metric/metric.py"
LORA_RANK   = 32  # 
MAX_LENGTH  = 1024
EPOCHS      = 1
LR          = 2e-4
RUN_METRIC  = True

"""
WHAT IS LORA ?_________
LORA (Low-Rank Adaptation) is a technique for fine-tuning large language models efficiently by introducing low-rank matrices into the model's architecture. 
the idea is to add trainable low-rank matrices to the existing weights of the model, allowing for efficient adaptation to new tasks without updating all
the original model parameters. 

the matrices of the layer of the model are decomposed into two smaller matrices:

                                                                        W = W + A  @ B
                                                                        where W (n,m) and A (n,r) and B (r,m) with r << n,m
                                                                        
the number 32 means that the rank of the low-rank matrices A and B is 32 which mean  : 

                                                                        W = W + (alpha/beta) A  @ B
                                                                        where W (n,m) and A (n,32) and B (32,m)
                                                                        
At the fine tuning stage, only the parameters of the low-rank matrices A and B are updated while the original weights W remain frozen.
"""


# ── dataset ──────────────────────────────────────────────────────────────────
class ReasoningDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.rows, self.tok, self.max_len = df.reset_index(drop=True), tokenizer, max_len
        # df.reset_index(drop=True) creates a new DataFrame with a new index starting from 0, and drops the old index. 

    def __len__(self):
        return len(self.rows)
        # get the number of rows in the dataframe, which is the number of samples in the dataset
        
    def __getitem__(self, idx):
        row    = self.rows.iloc[idx]
        prompt = str(row["prompt"]).strip() # get the prompt from the row and strip to not remove the space inside the prompt
        answer = str(row["answer"]).strip() # get the answer from the row and strip to not remove the space inside the answer
        if not (answer.startswith("\\boxed{") and answer.endswith("}")):
            answer = f"\\boxed{{{answer}}}" # ensure that the answer is in the format \boxed{answer} as required by the prompt

        user  = f"Solve the problem. Return only the final answer inside \\boxed{{}}.\n\n{prompt}\n\nAnswer:"
        full  = f"{user} {answer}{self.tok.eos_token}" # add the eos token at the end of the answer to indicate the end of the sequence
        
        # calculate the length of the prompt; why we add the answer to the prompt ? 
        # because the model needs to see the full input (prompt + answer) to learn how to generate the answer based on the prompt
        # but we only want to calculate the loss on the answer part, so we need to know how many tokens are in the prompt to ignore them in the loss calculation
        n_prompt = len(self.tok(user, truncation=True, max_length=self.max_len, add_special_tokens=True)["input_ids"])
        # this line will send only the tokens of the input and the answer
        
        # the below line will tokenize the full text (prompt + answer) and return the input ids and attention mask for the model
        enc = self.tok(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        # to get tokens that has the same size for training the tokens are padded so we add zero and then we ignore them we can spot them by  attention mask
        # attention mask is [1, 1, 1, 0, 0] where 1 means the token is part of the input and 0 means it's a padding token. 
        # ; the truncation will cut the input if it's too long
        ids, mask = enc["input_ids"][0], enc["attention_mask"][0] # 
        labels = ids.clone() # create a copy of the input ids to use as labels for the loss calculation
        labels[:n_prompt] = -100  # set the labels of the prompt tokens to -100 to ignore them in the loss calculation
        labels[mask == 0] = -100 # set the labels of the padding tokens to -100 to ignore them in the loss calculation
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}

# ── metric ───────────────────────────────────────────────────────────────────
def evaluate(eval_df):
    if not os.path.exists(METRIC_PATH):
        print(f"Skipping: metric.py not found at {METRIC_PATH}"); return None
    # Dynamically import the metric module from the specified path 
    spec = importlib.util.spec_from_file_location("nvidia_nemotron_metric", METRIC_PATH)
    metric = importlib.util.module_from_spec(spec); spec.loader.exec_module(metric)
    acc = metric.score(
        solution   = eval_df[["id", "answer"]].copy(),
        submission = pd.DataFrame({"id": eval_df["id"].tolist(), "prediction": [OUTPUT_DIR] * len(eval_df)}),
        row_id_column_name="id", max_lora_rank=LORA_RANK, debug=True,
    )
    print(f"accuracy: {acc:.6f}"); return acc

# ── main ─────────────────────────────────────────────────────────────────────
def main():
    if LORA_RANK > 32:
        raise ValueError("LORA_RANK must be <= 32.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16)
    model = get_peft_model(model, LoraConfig(
        r=LORA_RANK, lora_alpha=16,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    ))
    model.config.use_cache = False
    model.print_trainable_parameters()
    # df : is a pandas DataFrame that contains the training data loaded from the CSV file at TRAIN_PATH.
    # example : df = [ {"id": 1, "prompt": "What is 2 + 2?", "answer": "4"},
    #                {"id": 2, "prompt": "What is the capital of France?", "answer": "Paris"},
    #                ... ]
    # frac =1.0 means that we want to sample 100% of the data, random_state=42 is used to ensure that the shuffling is reproducible, and reset_index(drop=True)
    # is used to reset the index of the DataFrame after shuffling and dropping the old index.
    df       = pd.read_csv(TRAIN_PATH).sample(frac=1.0, random_state=42).reset_index(drop=True)
    split    = int(len(df) * 0.80)
    train_df, eval_df = df.iloc[:split].copy(), df.iloc[split:].copy()

    try:
        import bitsandbytes  # noqa: F401
        optim = "paged_adamw_8bit"
    except ImportError:
        optim = "adamw_torch"

    eval_key = "eval_strategy" if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else "evaluation_strategy"
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="/kaggle/working/nemotron_training_logs",
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            gradient_accumulation_steps=8, learning_rate=LR,
            bf16=True, optim=optim,
            logging_strategy="steps", logging_steps=10,
            save_strategy="epoch", report_to="tensorboard",logging_dir="/kaggle/working/logs",
            remove_unused_columns=False, gradient_checkpointing=True,
            **{eval_key: "epoch"},
        ),
        train_dataset=ReasoningDataset(train_df, tokenizer, MAX_LENGTH),
        eval_dataset =ReasoningDataset(eval_df,  tokenizer, MAX_LENGTH),
    )
    trainer.train()

    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Saved LoRA adapter to: {OUTPUT_DIR}")

    if RUN_METRIC:
        evaluate(eval_df)

if __name__ == "__main__":
    main()

/tmp/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


Epoch,Training Loss,Validation Loss
1,2.381883,0.344236


RuntimeError: on_train_begin must be called before on_evaluate

In [6]:

inputs = tokenizer("QUESTION  : 89+76= ? ", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tshape nameokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

QUESTION  : 89+76= ?  A) 135  B) 145  C) 155  D) 165

ANSWER : 165 "

Probably they want explanation? Or they posted the question. They


In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [2]:
print('Done.')
%load_ext tensorboard

Done.
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
